In [2]:
# update every year the initial nb of CR at the beginning of the year
from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np
import math
import os

# =========================
# 1. 读取 CSV
# =========================
input_file = "operations_CR_2005_2024.csv"

years = list(range(2010, 2025))
df = pd.read_csv(input_file, encoding="utf-8-sig")

for year in years:
    cr_col = f"Nb CR {year}"
    if cr_col not in df.columns:
        df[cr_col] = 0.0
    df[cr_col] = pd.to_numeric(
        df[cr_col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    ).fillna(0)

df.columns = (
    df.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
print(df.columns.tolist())


N = len(df)

# 转换数值列
for y in years:
    col = f"Nb CR {y}"
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )

mass_col = "Masse rejetée à chaque CR (kg)"
if mass_col not in df.columns:
    raise ValueError(f"找不到列: {mass_col}")

df[mass_col] = pd.to_numeric(
    df[mass_col].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

# 缺失值按 0 处理
df[mass_col] = df[mass_col].fillna(0)

# 检查 N Equipement
if "N Equipement" not in df.columns:
    raise ValueError("找不到列: N Equipement")

# =========================
# 2. 参数
# =========================
C_MANT = 48000
C_LEAK = 2520.0

equipment_count = len(df)

MAX_TASKS_PER_YEAR = max(1, math.floor(equipment_count * 0.061291843))
MAX_COST_PER_YEAR = equipment_count * 2700

# =========================
# 3. 初始化状态
# =========================
mass = df[mass_col].astype(float).to_numpy()

# 是否已经维修过
is_repaired = np.array([False] * N)

# 每个设备第一次被维修的年份
repair_year = np.array([np.nan] * N, dtype=float)

# 每个设备总泄漏量
equipment_total_leak = np.zeros(N)

# 每年决策记录
decision_records = []

# 每年汇总记录
year_summary_records = []

# =========================
# 4. 滚动窗口优化
# =========================
for year_idx, year in enumerate(years):
    cr_col = f"Nb CR {year}"
    if cr_col not in df.columns:
        raise ValueError(f"找不到列: {cr_col}")

    # -------------------------
    # 4.1 新窗口初始 CR 更新
    # 未维修设备：读取该年份 csv 原始值
    # 已维修设备：强制为 0
    # -------------------------
    current_cr = df[cr_col].fillna(0).astype(float).to_numpy()
    current_cr[is_repaired] = 0.0

    remaining_years = years[year_idx:]
    future_years_count = max(len(remaining_years) - 1, 0)

    # -------------------------
    # 4.2 计算当年实际泄漏
    # 当年泄漏 = 当前 CR × mass
    # -------------------------
    this_year_leak = current_cr * mass
    equipment_total_leak += this_year_leak

    # -------------------------
    # 4.3 建立 Gurobi 模型
    # x[i] = 1 表示在 year 年决定对设备 i 做 MANT
    # 该决策对下一窗口生效
    # -------------------------
    mdl = Model(f"rolling_mant_{year}")
    mdl.Params.OutputFlag = 1
    mdl.Params.MIPGap = 0.01
    mdl.Params.TimeLimit = 120

    x = mdl.addVars(N, vtype=GRB.BINARY, name="x")

    # 约束1：每年维修数量限制
    mdl.addConstr(
        quicksum(x[i] for i in range(N)) <= MAX_TASKS_PER_YEAR,
        name=f"max_tasks_{year}"
    )

    # 约束2：每年维修预算限制
    mdl.addConstr(
        quicksum(C_MANT * x[i] for i in range(N)) <= MAX_COST_PER_YEAR,
        name=f"max_cost_{year}"
    )

    # 已经维修过 or 当前 CR <= 0 的设备，不允许本年维修
    for i in range(N):
        if is_repaired[i] or current_cr[i] <= 0:
            mdl.addConstr(x[i] == 0, name=f"fix_zero_{year}_{i}")

    # -------------------------
    # 4.4 目标函数
    # 当前年泄漏已经发生，不能通过当年决策改变
    # 所以目标里只考虑未来年份 year+1 ~ 2024 的泄漏
    #
    # 如果今年不修：
    #   未来每年泄漏 = 当前假设 CR × mass
    # 如果今年修：
    #   下一窗口起 CR = 0，未来泄漏为 0
    # -------------------------
    action_cost = quicksum(C_MANT * x[i] for i in range(N))

    future_leak_cost = quicksum(
        (current_cr[i] * mass[i] * future_years_count) * (1 - x[i]) * C_LEAK
        for i in range(N)
        if (not is_repaired[i]) and current_cr[i] > 0
    )

    mdl.setObjective(action_cost + future_leak_cost, GRB.MINIMIZE)
    mdl.optimize()

    # -------------------------
    # 4.5 读取决策结果
    # -------------------------
    if mdl.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        print(f"Year {year}: 优化失败，status={mdl.status}")
        selected_this_year = []
    else:
        selected_this_year = [i for i in range(N) if x[i].X > 0.5]

    # 记录设备级决策
    for i in selected_this_year:
        if np.isnan(repair_year[i]):
            repair_year[i] = year

    decision_records.extend([
        {
            "year": year,
            "N Equipement": df.iloc[i]["N Equipement"],
            "current_cr": current_cr[i],
            "mass_per_cr": mass[i],
            "this_year_leak": this_year_leak[i],
            "selected_for_MANT": 1,
            "future_years_count": future_years_count
        }
        for i in selected_this_year
    ])

    # 年度汇总
    year_summary_records.append({
        "year": year,
        "equipment_with_positive_cr": int((current_cr > 0).sum()),
        "selected_count": len(selected_this_year),
        "year_leak_kg": float(this_year_leak.sum()),
        "year_action_cost": len(selected_this_year) * C_MANT
    })

    # -------------------------
    # 4.6 更新维修状态
    # 注意：这里只更新 is_repaired
    # 下一窗口开始时会重新从 csv 读新年份 CR，
    # 然后对已维修设备强制置 0
    # -------------------------
    for i in selected_this_year:
        is_repaired[i] = True

    print(f"Year {year}: selected {len(selected_this_year)} equipments for MANT")

# =========================
# 5. 输出设备级结果
# =========================
result_df = df[["N Equipement"]].copy()
result_df["repair_year"] = repair_year
result_df["repaired"] = is_repaired.astype(int)
result_df["equipment_total_leak"] = equipment_total_leak
result_df["mass_per_cr_kg"] = df[mass_col]

# 可选：保留每年原始 CR 方便核对
for y in years:
    col = f"Nb CR {y}"
    if col in df.columns:
        result_df[col] = df[col]

# =========================
# 6. 输出决策结果
# =========================
decision_df = pd.DataFrame(decision_records)
if decision_df.empty:
    decision_df = pd.DataFrame(columns=[
        "year", "N Equipement", "current_cr", "mass_per_cr",
        "this_year_leak", "selected_for_MANT", "future_years_count"
    ])

# =========================
# 7. 输出年度汇总
# =========================
year_summary_df = pd.DataFrame(year_summary_records)

# =========================
# 8. 总汇总
# =========================
summary_df = pd.DataFrame({
    "total_leak_kg": [result_df["equipment_total_leak"].sum()],
    "total_equipment": [N],
    "repaired_equipment_count": [result_df["repaired"].sum()],
    "total_action_cost": [result_df["repaired"].sum() * C_MANT],
    "C_MANT": [C_MANT],
    "C_LEAK": [C_LEAK],
    "MAX_TASKS_PER_YEAR": [MAX_TASKS_PER_YEAR],
    "MAX_COST_PER_YEAR": [MAX_COST_PER_YEAR]
})

# =========================
# 9. 保存文件
# =========================
result_df.to_csv(
    os.path.join("opt/gurobi_rolling_mant_result.csv"),
    index=False,
    encoding="utf-8-sig"
)

decision_df.to_csv(
    os.path.join("opt/gurobi_rolling_mant_decisions.csv"),
    index=False,
    encoding="utf-8-sig"
)

year_summary_df.to_csv(
    os.path.join("opt/gurobi_rolling_mant_year_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    os.path.join("opt/gurobi_rolling_mant_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 10. 控制台输出
# =========================
print("\n========== RESULT SUMMARY ==========")
print(summary_df)

print("\n========== YEAR SUMMARY ==========")
print(year_summary_df)

print("\n========== EQUIPMENT RESULT PREVIEW ==========")
print(result_df.head(10))

print("\n========== DECISION PREVIEW ==========")
print(decision_df.head(10))

['N Equipement', 'Date MES CSEM', 'Masse de Gaz CSEM (kg)', 'Pression Nominale (bar)', '% de la masse de gaz rejetée par CR', 'Masse rejetée à chaque CR (kg)', 'Nb CR 2010', 'Nb CR 2011', 'Nb CR 2012', 'Nb CR 2013', 'Nb CR 2014', 'Nb CR 2015', 'Nb CR 2016', 'Nb CR 2017', 'Nb CR 2018', 'Nb CR 2019', 'Nb CR 2020', 'Nb CR 2021', 'Nb CR 2022', 'Nb CR 2023', 'Nb CR 2024', "Nb atteinte critère d'exploit 12 mois", 'Masse rejetée 1 an glissant (kg)', 'Date du dernier OT EVA realized', 'Nature du dernier OT EVA realized', 'Date Planif Prochain OT', 'Nature Prochain OT', 'La dernière Fuite est-elle traitée?', 'Eligible EVA']
Set parameter Username
Set parameter LicenseID to value 2773682
Academic license - for non-commercial use only - expires 2027-02-02
Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 120
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimi